# Customer Retention Intelligence — Iteration 1

This notebook builds a **working baseline customer retention model**.

By the end, you will:
- load the customer dataset
- inspect and clean the data
- prepare features
- train a baseline classification model
- evaluate it with metrics better than accuracy alone
- save the trained model for later iterations

> Dataset expected at: `../data/Customer_Churn_Dataset.xlsx`  
> Sheet expected: `01 Churn-Dataset`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    RocCurveDisplay
)

import joblib


## 1. Load the dataset

In [ ]:
df = pd.read_excel(
    "../data/Customer_Churn_Dataset.xlsx",
    sheet_name="01 Churn-Dataset"
)

df.head()


## 2. Basic data inspection

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
df.info()


In [ ]:
print(df["Churn"].value_counts())
print()
print(df["Churn"].value_counts(normalize=True))


In [ ]:
df.isnull().sum().sort_values(ascending=False)


## 3. Inspect data types

`TotalCharges` sometimes appears as text in versions of this dataset, so we check and clean it safely.


In [ ]:
df.dtypes


In [ ]:
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print(df["TotalCharges"].head(20))


## 4. Clean the dataset

In [ ]:
# Convert TotalCharges to numeric.
# Any non-numeric values become NaN.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing values in TotalCharges after conversion:")
print(df["TotalCharges"].isnull().sum())


In [ ]:
# Fill missing TotalCharges values using the median.
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

print("Remaining missing values:")
print(df.isnull().sum().sum())


## 5. Remove columns that should not be used as predictors

`customerID` identifies a customer but does not describe customer behavior, so we remove it from the model.


In [ ]:
df_model = df.drop(columns=["customerID"]).copy()

df_model.head()


## 6. Convert the target to numeric

In [ ]:
df_model["Churn"] = df_model["Churn"].map({
    "No": 0,
    "Yes": 1
})

df_model["Churn"].value_counts()


## 7. Separate features and target

In [ ]:
X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


## 8. Identify numerical and categorical features

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)


## 9. Train-test split

We use `stratify=y` so the churn proportion stays similar in both train and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nTraining churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())


## 10. Build preprocessing pipeline

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


## 11. Build the baseline model

We start with **Logistic Regression** because it is:
- strong as a baseline
- fast
- easy to explain
- commonly used for churn/retention problems


In [ ]:
baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

baseline_model


## 12. Train the model

In [ ]:
baseline_model.fit(X_train, y_train)

print("Model training complete.")


## 13. Make predictions

In [ ]:
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

print("Predictions generated:", len(y_pred))


## 14. Evaluate the model

Because churn is imbalanced, we look at:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"ROC-AUC:   {roc_auc:.3f}")


In [ ]:
print(classification_report(y_test, y_pred))


## 15. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Stay", "Actual Churn"],
    columns=["Predicted Stay", "Predicted Churn"]
)

cm_df


## 16. ROC curve

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("Baseline Logistic Regression ROC Curve")
plt.show()


## 17. Create customer-level retention risk output

This is the first step toward our final product.


In [ ]:
results = X_test.copy()

results["actual_churn"] = y_test.values
results["predicted_churn"] = y_pred
results["churn_probability"] = y_prob

results["risk_level"] = pd.cut(
    results["churn_probability"],
    bins=[-0.01, 0.30, 0.60, 1.00],
    labels=["Low", "Medium", "High"]
)

results[
    ["churn_probability", "risk_level", "actual_churn", "predicted_churn"]
].sort_values(
    by="churn_probability",
    ascending=False
).head(20)


## 18. Save the baseline model

In [ ]:
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    baseline_model,
    "../models/baseline_retention_model.joblib"
)

print("Saved to ../models/baseline_retention_model.joblib")


# Iteration 1 complete

At this point we have a working product that can:

1. accept customer characteristics
2. preprocess numerical and categorical variables
3. estimate customer churn probability
4. assign a basic Low / Medium / High retention-risk level
5. save the trained ML pipeline

### Next iteration
We will improve the model by adding:
- stronger exploratory data analysis
- feature engineering
- multiple models
- cross-validation
- better threshold selection
- model comparison
